# Инференс ruBERT с разной квантизацией

Ноутбук выбирает 10 случайных новостей из `data/splits/test_relabeled.json`,
прогоняет их через выбранную ONNX Runtime-модель и показывает категории с
вероятностями в HTML-карточках вместе с реальными метками из `target` и
сверкой предсказаний. Доступны исходная FP32-модель, INT8 и INT4.

Для интерактивного dropdown установите зависимости из
`requirements-rubert-onnx-notebook.txt`. Без `ipywidgets` последняя ячейка
выполнит демонстрационный прогон INT8 автоматически.

In [4]:
from __future__ import annotations

import html as html_lib
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
from IPython.display import HTML, clear_output, display

# Resolve the repository root whether Jupyter was started from ml/ or ml/notebooks/.
PROJECT_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / 'rubert_multilabel').is_dir()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT / 'experiments'))
from onnx_rubert_classifier import RubertOnnxClassifier

DATA_PATH = PROJECT_ROOT / 'data' / 'splits' / 'test_relabeled.json'
MODEL_DIRS = {
    'FP32 — без квантизации': PROJECT_ROOT / 'rubert_multilabel' / 'onnx',
    'INT8': PROJECT_ROOT / 'rubert_multilabel' / 'onnx-int8',
    'INT4': PROJECT_ROOT / 'rubert_multilabel' / 'onnx-int4',
}

if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Не найден датасет: {DATA_PATH}')
missing_models = [str(path) for path in MODEL_DIRS.values() if not (path / 'model.onnx').is_file()]
if missing_models:
    raise FileNotFoundError(f'Не найдены ONNX-модели: {missing_models}')

NEWS = json.loads(DATA_PATH.read_text(encoding='utf-8'))
if len(NEWS) < 10:
    raise ValueError(f'В датасете только {len(NEWS)} записей, нужно минимум 10')

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

CLASSIFIERS: dict[str, RubertOnnxClassifier] = {}

def get_classifier(model_name: str) -> RubertOnnxClassifier:
    if model_name not in MODEL_DIRS:
        raise ValueError(f'Неизвестная модель: {model_name}')
    if model_name not in CLASSIFIERS:
        CLASSIFIERS[model_name] = RubertOnnxClassifier(MODEL_DIRS[model_name])
    return CLASSIFIERS[model_name]

print(f'Проект: {PROJECT_ROOT}')
print(f'Новости для теста: {len(NEWS)}')
print('Модели:', ', '.join(MODEL_DIRS))

Проект: /Users/alexpechul/study/alfa-hackathon/ml
Новости для теста: 500
Модели: FP32 — без квантизации, INT8, INT4


In [5]:
CARD_CSS = '''
<style>
.rubert-report {
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
  color: #182230; background: #f5f7fb; padding: 20px; border-radius: 14px;
}
.rubert-report h2 { margin: 0 0 4px; font-size: 22px; }
.rubert-report .subtitle { color: #617086; margin: 0 0 18px; font-size: 13px; }
.rubert-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(min(100%, 330px), 1fr)); gap: 14px; }
.rubert-card { background: #fff; border: 1px solid #dce3ed; border-radius: 10px; overflow: hidden; box-shadow: 0 2px 8px rgba(28, 43, 64, .06); }
.rubert-card header { display: flex; justify-content: space-between; gap: 12px; padding: 13px 15px; background: #eef3fa; border-bottom: 1px solid #dce3ed; }
.rubert-card .number { font-weight: 700; color: #2457a6; }
.rubert-card .post-id { color: #738198; font-size: 12px; overflow-wrap: anywhere; text-align: right; }
.rubert-card .text { max-height: 180px; overflow-y: auto; padding: 14px 15px; line-height: 1.48; font-size: 14px; white-space: normal; }
.rubert-card section { padding: 0 15px 15px; }
.rubert-card h3 { margin: 0 0 9px; font-size: 12px; color: #617086; text-transform: uppercase; letter-spacing: 0; }
.labels { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 14px; }
.label { display: inline-flex; gap: 6px; align-items: center; padding: 6px 8px; border-radius: 7px; background: #e5f5ed; color: #17643f; font-size: 12px; line-height: 1.2; }
.label .score { font-weight: 700; white-space: nowrap; }
.label.truth { background: #e7efff; color: #244a8a; }
.comparison { display: grid; gap: 7px; margin: -4px 0 14px; font-size: 12px; }
.comparison .status { font-weight: 700; }
.comparison .exact { color: #17643f; }
.comparison .different { color: #a64b12; }
.comparison .group { display: grid; grid-template-columns: 90px 1fr; gap: 8px; align-items: start; color: #617086; }
.comparison .group strong { color: #465367; }
.comparison .tag { color: #182230; overflow-wrap: anywhere; }
.comparison .matched { color: #17643f; }
.comparison .missed { color: #a64b12; }
.comparison .extra { color: #a13c46; }
.empty-inline { color: #8995a7; }
.empty { color: #8995a7; font-size: 13px; margin-bottom: 14px; }
.top-table { width: 100%; border-collapse: collapse; font-size: 12px; }
.top-table td { padding: 5px 0; border-top: 1px solid #edf0f5; vertical-align: top; }
.top-table td:last-child { text-align: right; color: #2457a6; font-weight: 700; white-space: nowrap; padding-left: 8px; }
.bar { height: 5px; background: #e7ecf4; border-radius: 3px; margin-top: 4px; overflow: hidden; }
.bar span { display: block; height: 100%; background: #4c78c2; border-radius: 3px; }
.rubert-note { color: #617086; font-size: 12px; margin-top: 16px; }
</style>
'''

def _escape_text(value: object) -> str:
    return html_lib.escape(str(value)).replace('\n', '<br>')

def _percent(value: float) -> str:
    return f'{value * 100:.1f}%'

def _label_tags(labels: list[str], variant: str) -> str:
    if not labels:
        return "<span class='empty-inline'>—</span>"
    return ''.join(
        f"<span class='tag {variant}'>{html_lib.escape(label)}</span>"
        for label in labels
    )

def choose_news(seed: int = 42, count: int = 10) -> list[dict]:
    rng = random.Random(seed)
    return rng.sample(NEWS, k=min(count, len(NEWS)))

def render_inference(model_name: str = 'INT8', seed: int = 42, count: int = 10, top_k: int = 5) -> HTML:
    classifier = get_classifier(model_name)
    selected_news = choose_news(seed=seed, count=count)
    texts = [row['text'] for row in selected_news]
    started = time.perf_counter()
    probabilities = classifier.predict_proba(texts)
    elapsed = time.perf_counter() - started

    cards: list[str] = []
    for card_number, (row, scores) in enumerate(zip(selected_news, probabilities, strict=True), 1):
        ranking = np.argsort(scores)[::-1]
        predicted = [
            (classifier.labels[index], float(scores[index]), float(classifier.thresholds[index]))
            for index in ranking
            if scores[index] >= classifier.thresholds[index]
        ]
        predicted_names = [label for label, _score, _threshold in predicted]
        target = row.get('target') or {}
        true_labels = [
            label for label in classifier.labels
            if target.get(label, 0)
        ]
        true_set = set(true_labels)
        predicted_set = set(predicted_names)
        matched = [label for label in classifier.labels if label in true_set & predicted_set]
        missed = [label for label in classifier.labels if label in true_set - predicted_set]
        extra = [label for label in classifier.labels if label in predicted_set - true_set]
        exact_match = not missed and not extra
        if predicted:
            label_html = ''.join(
                f"<span class='label'>{html_lib.escape(label)} <span class='score'>{_percent(score)}</span></span>"
                for label, score, _threshold in predicted
            )
        else:
            label_html = "<div class='empty'>Нет категорий выше индивидуальных порогов</div>"

        comparison_status = (
            "<span class='status exact'>Полное совпадение</span>"
            if exact_match else "<span class='status different'>Есть расхождения</span>"
        )
        true_html = ''.join(
            f"<span class='label truth'>{html_lib.escape(label)}</span>"
            for label in true_labels
        ) or "<div class='empty'>Нет реальных категорий</div>"
        comparison_html = (
            f"<div class='comparison'>{comparison_status}"
            f"<div class='group'><strong>Совпали</strong><div>{_label_tags(matched, 'matched')}</div></div>"
            f"<div class='group'><strong>Пропущены</strong><div>{_label_tags(missed, 'missed')}</div></div>"
            f"<div class='group'><strong>Лишние</strong><div>{_label_tags(extra, 'extra')}</div></div></div>"
        )

        top_rows = []
        for index in ranking[:top_k]:
            label = html_lib.escape(classifier.labels[index])
            score = float(scores[index])
            top_rows.append(
                f"<tr><td>{label}<div class='bar'><span style='width:{score * 100:.2f}%'></span></div></td>"
                f"<td>{_percent(score)}</td></tr>"
            )

        post_id = html_lib.escape(str(row.get('post_id', 'без id')))
        cards.append(
            f"""<article class='rubert-card'>
              <header><span class='number'>Новость {card_number}</span><span class='post-id'>{post_id}</span></header>
              <div class='text'>{_escape_text(row['text'])}</div>
              <section><h3>Реальные категории</h3><div class='labels'>{true_html}</div>
              <h3>Категории выше порога</h3><div class='labels'>{label_html}</div>
              <h3>Сверка с разметкой</h3>{comparison_html}
              <h3>Топ-{top_k} вероятностей</h3><table class='top-table'>{''.join(top_rows)}</table></section>
            </article>"""
        )

    title = html_lib.escape(model_name)
    return HTML(
        CARD_CSS
        + f"<div class='rubert-report'><h2>ruBERT: {title}</h2>"
        + f"<p class='subtitle'>10 случайных новостей · seed={seed} · batch={len(texts)} · инференс {elapsed:.2f} с</p>"
        + f"<div class='rubert-grid'>{''.join(cards)}</div>"
        + "<div class='rubert-note'>Вероятности — sigmoid от logits; категории выбираются с индивидуальными порогами из thresholds.json.</div></div>"
    )

In [ ]:
# Интерактивный выбор модели. Нажмите кнопку после выбора FP32, INT8 или INT4.
if widgets is None:
    print('ipywidgets не установлен; выполняю демонстрационный прогон INT8.')
    display(render_inference('INT8', seed=42, count=10))
else:
    model_selector = widgets.Dropdown(
        options=list(MODEL_DIRS),
        value='INT8',
        description='Модель:',
        layout=widgets.Layout(width='360px'),
    )
    seed_selector = widgets.IntText(
        value=42,
        description='Seed:',
        layout=widgets.Layout(width='180px'),
    )
    run_button = widgets.Button(
        description='Показать прогнозы',
        button_style='primary',
        icon='play',
    )
    output = widgets.Output()

    def run_inference(_=None):
        with output:
            clear_output(wait=True)
            display(render_inference(model_selector.value, seed=seed_selector.value, count=10))

    run_button.on_click(run_inference)
    display(widgets.VBox([widgets.HBox([model_selector, seed_selector, run_button]), output]))
    run_inference()